In [14]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
import os

print('Libraries loaded')

Libraries loaded


## Step 1: Load the manually labelled data

In [15]:
df = pd.read_csv('../data/raw/reviews_labelled.csv')
print(f'Loaded {len(df)} manually labelled reviews')
print(f'\nCategory label distribution:')
print(df['category_name'].value_counts())
df.head()

Loaded 2209 manually labelled reviews

Category label distribution:
category_name
Positive Praise    832
Bug Report         753
UX Feedback        439
Feature Request    185
Name: count, dtype: int64


,review_id,review_text,star_rating,date,app_id,app_category,category_label,category_name
0,570b5412-5b91-48bb-8c06-963879515606,Android (Oxygen OS 16): home screen widget is ...,1,2026-06-16 18:22:10,com.todoist,productivity,0,Bug Report
1,515ec419-cce5-41da-aaff-5779f2031805,EDIT: I reached out to support as advised in t...,2,2026-06-16 16:36:06,com.todoist,productivity,1,Feature Request
2,75f42825-a30b-4237-9afe-c051a23b6e63,"not working on oneplus 9rt , can't login",1,2026-06-16 16:35:08,com.todoist,productivity,0,Bug Report
3,aa395a6b-07b5-422e-960e-e1317436b2ac,Yesterday I mentioned that a task disappeared ...,1,2026-06-16 03:31:49,com.todoist,productivity,0,Bug Report
4,41f403b4-c9a4-4bc3-bc29-756df20edac7,"Installed it, created an account, realized I n...",1,2026-06-15 19:18:12,com.todoist,productivity,2,UX Feedback


## Step 2: Clean review text

In [16]:
def clean_text(text):
    
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'[^a-z0-9\s!?.,]', ' ', text)    # keep basic punctuation
    text = re.sub(r'\s+', ' ', text).strip()         # remove extra spaces
    return text

df['clean_text'] = df['review_text'].apply(clean_text)

# Remove empty after cleaning
df = df[df['clean_text'].str.len() > 0]

print(f'Reviews after cleaning: {len(df)}')
print('\nExample — before and after:')
for i in range(3):
    print(f'  BEFORE: {df["review_text"].iloc[i][:100]}')
    print(f'  AFTER:  {df["clean_text"].iloc[i][:100]}')
    print()

Reviews after cleaning: 2200

Example — before and after:
  BEFORE: Android (Oxygen OS 16): home screen widget is broken after the last update.
  AFTER:  android oxygen os 16 home screen widget is broken after the last update.

  BEFORE: EDIT: I reached out to support as advised in the response but they were not willing to do anything. 
  AFTER:  edit i reached out to support as advised in the response but they were not willing to do anything. i

  BEFORE: not working on oneplus 9rt , can't login
  AFTER:  not working on oneplus 9rt , can t login



## Step 3: Derive sentiment labels from star rating

In [17]:
def assign_sentiment(star_rating):
    if star_rating <= 2:
        return 0  # Negative
    elif star_rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['sentiment_label'] = df['star_rating'].apply(assign_sentiment)
df['sentiment_name'] = df['sentiment_label'].map({0: 'Negative', 1: 'Neutral', 2: 'Positive'})

print('Sentiment distribution:')
print(df['sentiment_name'].value_counts())

Sentiment distribution:
sentiment_name
Negative    1045
Positive     990
Neutral      165
Name: count, dtype: int64


## Step 4: Category labels


In [18]:
# Validate that manual labels are complete and consistent
assert df['category_label'].isnull().sum() == 0, 'Found rows with missing category_label'
assert df['category_name'].isnull().sum() == 0, 'Found rows with missing category_name'

label_map_check = df.groupby('category_label')['category_name'].unique()
print('Label -> name mapping found in data:')
print(label_map_check)

print('\nCategory distribution (manually labelled):')
print(df['category_name'].value_counts())

label_counts = df['category_label'].value_counts()
imbalance_ratio = label_counts.max() / label_counts.min()
print(f'\nImbalance ratio (majority/minority): {imbalance_ratio:.2f}x')

Label -> name mapping found in data:
category_label
0         [Bug Report]
1    [Feature Request]
2        [UX Feedback]
3    [Positive Praise]
Name: category_name, dtype: object

Category distribution (manually labelled):
category_name
Positive Praise    829
Bug Report         748
UX Feedback        438
Feature Request    185
Name: count, dtype: int64

Imbalance ratio (majority/minority): 4.48x


## Step 5: Manual labelling check


In [19]:
# Print sample for manual checking
sample = df.sample(20, random_state=42)[['clean_text', 'category_name', 'sentiment_name']]
for i, row in sample.iterrows():
    print(f'Review: {row["clean_text"][:150]}')
    print(f'  Category: {row["category_name"]} | Sentiment: {row["sentiment_name"]}')
    print()

Review: i liked calm app a lot and can pay in monthly payments of 6.66 a month 4 a yearly sub. 4 a yr. starting on may 4 th. for sure. i ve asked numerous tim
  Category: Positive Praise | Sentiment: Positive

Review: excellent app so far! more lebron james please!
  Category: Positive Praise | Sentiment: Positive

Review: i need to pay to use the most unsafe app, keep your kids away from telegram it s dangerous
  Category: UX Feedback | Sentiment: Negative

Review: this app is really good
  Category: Positive Praise | Sentiment: Positive

Review: instagram is my favourite app, but for the love of god, when are you going to give us an edit option? only once your comment is posted do you get to s
  Category: UX Feedback | Sentiment: Neutral

Review: the app keeps logging me out and when i try to sign in, it s always a problem. then i have to reset my password over and over! extremely frustrating!
  Category: Bug Report | Sentiment: Negative

Review: i don t know why i can t install it!!

## Step 6: Train / Validation / Test split

In [20]:
# 70% train, 15% validation, 15% test
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['category_label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['category_label']
)

print(f'Train size:      {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)')
print(f'Validation size: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test size:       {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)')

print('\nCategory distribution in train set:')
print(train_df['category_name'].value_counts())

Train size:      1540 (70.0%)
Validation size: 330 (15.0%)
Test size:       330 (15.0%)

Category distribution in train set:
category_name
Positive Praise    580
Bug Report         524
UX Feedback        307
Feature Request    129
Name: count, dtype: int64


## Step 7: Save processed data

In [21]:
os.makedirs('../data/processed', exist_ok=True)

train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)
df.to_csv('../data/processed/full_labelled.csv', index=False)

print('Saved:')
print('  ../data/processed/train.csv')
print('  ../data/processed/val.csv')
print('  ../data/processed/test.csv')
print('  ../data/processed/full_labelled.csv')


Saved:
  ../data/processed/train.csv
  ../data/processed/val.csv
  ../data/processed/test.csv
  ../data/processed/full_labelled.csv
